# 05 · Cassava as a wheat-substitution lever (GROW production lens)

Notebooks 01–04 asked *what Indonesia grows and how it shifted*. This one makes the GROW data
serve the team's actual thesis: **Indonesia is the world's #1 wheat importer and grows no wheat at all —
can domestically-grown cassava (via MOCAF, fermented cassava flour) realistically offset part of that?**

The whole MOCAF argument rests on one unchecked production-side premise: *that Indonesia grows enough
cassava, and that the base is healthy enough, to matter.* This notebook checks it, entirely within GROW
data (QCL production/area/yield), and produces the flour-equivalent number the recommendation needs.

**Runs against the same setup as 01–04** — `src.load.load_indonesia`, `src.clean`, `src.viz`. No new download.

Sections:
1. Setup
2. The structural zero — wheat vs domestic staples
3. Cassava production / area / yield, 2010–latest — is the base growing or shrinking?
4. What's driving cassava's trajectory — area vs yield decomposition
5. Cassava yield gap vs the global frontier — headroom without new cropland
6. Flour-equivalent: how much wheat *could* cassava offset?
7. Findings + recommendation feed for notebook 04

## 1. Setup

In [ ]:
import sys; sys.path.append('..')
import pandas as pd, numpy as np, matplotlib.pyplot as plt
from src.load import load_indonesia
from src.clean import drop_item_aggregates
from src import viz

qcl = load_indonesia('QCL')
LATEST = qcl['year'].max()
BASE   = qcl['year'].min()
print(f'QCL Indonesia: years {BASE}–{LATEST}')

# FAOSTAT item name for cassava in QCL is 'Cassava, fresh' (confirmed in nb 02/03).
# Guard against a naming drift between FAOSTAT releases.
CASSAVA = 'Cassava, fresh'
assert CASSAVA in set(qcl['Item']), \
    f"'{CASSAVA}' not found — candidates: {[i for i in qcl['Item'].unique() if 'assava' in i]}"

## 2. The structural zero — wheat vs the domestic staples

The root of the import dependence, stated as a GROW fact: Indonesia produces **no wheat** (it has no QCL
wheat records at all — not near-zero, literally absent), while it grows large volumes of every other
calorie staple. That contrast is *why* the substitution question has to be asked on the domestic side.

In [ ]:
staples = ['Wheat', 'Rice', 'Maize (corn)', CASSAVA, 'Soya beans']
prod_latest = qcl[(qcl['Element'] == 'Production') & (qcl['year'] == LATEST)]
prod_latest = drop_item_aggregates(prod_latest)

rows = []
for s in staples:
    v = prod_latest[prod_latest['Item'] == s]['value'].sum()
    present = s in set(qcl['Item'])
    rows.append({'staple': s,
                 'production_Mt_' + str(LATEST): v / 1e6,
                 'in_QCL_at_all': present})
staple_tbl = pd.DataFrame(rows)
print(staple_tbl.to_string(index=False))
print()
print(f"Wheat records anywhere in Indonesia QCL, any year: {(qcl['Item'] == 'Wheat').sum()}")
print('  -> zero = the structural, production-side root of Indonesia''s wheat-import dependence.')

In [ ]:
# Visual: domestic staple production, with wheat pinned at 0 to make the contrast the takeaway.
plot_s = staple_tbl.set_index('staple')[f'production_Mt_{LATEST}'].sort_values()
fig, ax = plt.subplots(figsize=(9, 4.5))
colors = ['#c0392b' if s == 'Wheat' else '#2a78d6' for s in plot_s.index]
ax.barh(plot_s.index, plot_s.values, color=colors)
ax.set_xlabel(f'Domestic production, {LATEST} (million tonnes)')
ax.set_title('Indonesia grows every calorie staple at scale — except wheat (zero)')
for i, (name, val) in enumerate(plot_s.items()):
    ax.text(val, i, f'  {val:,.1f} Mt' + ('  ← imported wholesale' if name == 'Wheat' else ''),
            va='center', fontsize=9)
fig.tight_layout()
viz.save(fig, 'idn_staples_vs_wheat_zero')
plt.show()

## 3. Is the cassava base growing or shrinking?

MOCAF is the proposed solution — but a solution needs raw material. Notebook 03's 2015→2024
decomposition already hinted cassava **area** fell. Here's the full production / area / yield picture.

In [ ]:
cas = qcl[(qcl['Item'] == CASSAVA) &
          (qcl['Element'].isin(['Production', 'Area harvested', 'Yield']))]
cpiv = cas.pivot_table(index='year', columns='Element', values='value').sort_index()
yield_unit = cas[cas['Element'] == 'Yield']['Unit'].iloc[0]
print(f'Yield unit as stored: {yield_unit}  (FAOSTAT QCL yield is kg/ha)')
cpiv.round(0)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
ax2 = ax.twinx()
ax.plot(cpiv.index, cpiv['Area harvested'], color='#2a78d6', marker='o', label='Area harvested (ha)')
ax2.plot(cpiv.index, cpiv['Yield'], color='#eb6834', marker='s', label='Yield (kg/ha)')
ax.set_ylabel('Area harvested (ha)', color='#2a78d6')
ax2.set_ylabel(f'Yield ({yield_unit})', color='#eb6834')
ax.set_title('Cassava: area harvested vs yield over time')
fig.tight_layout()
viz.save(fig, 'idn_cassava_area_vs_yield')
plt.show()

## 4. What's driving cassava's trajectory? (area vs yield — same method as rice in nb 02)

In [ ]:
y0, y1 = cpiv.index.min(), cpiv.index.max()
chg = {}
for col in ['Production', 'Area harvested', 'Yield']:
    v0, v1 = cpiv[col].loc[y0], cpiv[col].loc[y1]
    chg[col] = 100 * (v1 - v0) / v0
    print(f'{col:16s}: {v0:>14,.0f} ({y0}) -> {v1:>14,.0f} ({y1})  ({chg[col]:+.1f}%)')

corr_area  = cpiv['Area harvested'].corr(cpiv['Production'])
corr_yield = cpiv['Yield'].corr(cpiv['Production'])
print(f'\ncorr(area, production)  = {corr_area:+.2f}')
print(f'corr(yield, production) = {corr_yield:+.2f}')

if chg['Production'] < 0 and chg['Area harvested'] < 0:
    driver = 'area contraction' if abs(chg['Area harvested']) > abs(chg['Yield']) else 'yield weakness'
    verdict = f'Cassava production is DECLINING, driven mainly by {driver}.'
elif chg['Production'] >= 0 and chg['Yield'] > abs(chg['Area harvested']):
    verdict = 'Cassava production holding/growing on YIELD despite area loss — intensification.'
else:
    verdict = 'Mixed — read the numbers above directly.'
print('\nVERDICT:', verdict)
print('IMPLICATION: MOCAF scale-up cannot assume a static raw-material base — the'
      ' cassava supply itself is a variable, and (if area-driven) competes with the'
      ' same land-conversion pressure flagged in the deck.')

## 5. Cassava yield gap — is there headroom *without* new cropland?

If Indonesian cassava yield sits well below the global frontier, substitution capacity can grow from
**yield**, not just area — the more sustainable lever, and one that sidesteps the land-conversion problem.
We compare Indonesia against the top cassava producers using the full (unfiltered) QCL, which includes
all countries.

> Note: `load_indonesia` returns Indonesia-only rows. For the cross-country frontier we need the
> multi-country QCL. If your repo exposes a full loader (e.g. `load_dataset('QCL')` / `load_qcl_long()`),
> use it here; the cell tries the common names and degrades gracefully if none is present.

In [ ]:
qcl_all = None
for loader_name in ['load_dataset', 'load_qcl_long', 'load_qcl', 'load_all']:
    try:
        import src.load as L
        if hasattr(L, loader_name):
            fn = getattr(L, loader_name)
            try:
                qcl_all = fn('QCL')
            except TypeError:
                qcl_all = fn()
            print(f'Loaded full QCL via src.load.{loader_name}(): {qcl_all.shape[0]:,} rows')
            break
    except Exception as e:
        print(f'{loader_name} failed: {e}')

if qcl_all is None:
    print('No full-QCL loader found. Fill FRONTIER_YIELD_KG_HA below from FAOSTAT manually'
          ' (QCL, Yield, Cassava, latest year) for Thailand/India/Nigeria, or point this'
          ' cell at your grow-eda cache, e.g. pd.read_parquet("../grow-eda/data/cache/QCL_long.parquet").')

In [ ]:
idn_yield_kg = cpiv['Yield'].loc[LATEST]  # kg/ha

if qcl_all is not None:
    ac = qcl_all[(qcl_all['Item'] == CASSAVA) &
                 (qcl_all['Element'] == 'Yield') &
                 (qcl_all['year'] == LATEST)].copy()
    ac = drop_item_aggregates(ac)
    ac = ac[ac['Area Code'] < 5000] if 'Area Code' in ac.columns else ac   # drop region aggregates
    top_yield = ac.groupby('Area')['value'].mean().sort_values(ascending=False)
    frontier = top_yield.head(5)
    print('Top-5 cassava yields worldwide (kg/ha, %s):' % LATEST)
    print(frontier.round(0))
    FRONTIER_YIELD_KG_HA = float(frontier.iloc[0])
else:
    # Manual fallback — FAOSTAT QCL cassava yield, recent frontier ~ India/Indonesia lead ~ 25–40 t/ha.
    FRONTIER_YIELD_KG_HA = 40000.0   # <-- replace with the real frontier from the printout above
    print(f'Using manual frontier placeholder: {FRONTIER_YIELD_KG_HA:,.0f} kg/ha — REPLACE with real value.')

gap_pct = 100 * (FRONTIER_YIELD_KG_HA - idn_yield_kg) / FRONTIER_YIELD_KG_HA
print(f'\nIndonesia cassava yield ({LATEST}): {idn_yield_kg:,.0f} kg/ha')
print(f'Frontier yield:                    {FRONTIER_YIELD_KG_HA:,.0f} kg/ha')
print(f'Yield gap:                         {gap_pct:+.0f}%  (headroom to frontier)')

# Potential extra tonnage if Indonesia closed the gap on its CURRENT area (no new cropland):
cur_area = cpiv['Area harvested'].loc[LATEST]
extra_t_if_frontier = (FRONTIER_YIELD_KG_HA - idn_yield_kg) * cur_area / 1000.0  # kg->t
print(f'\nIf the gap closed on today''s {cur_area:,.0f} ha: +{extra_t_if_frontier/1e6:,.1f} Mt cassava,'
      ' no new land.')

## 6. Flour-equivalent — how much wheat *could* cassava offset?

The number the DRAFT blending-matrix slide needs. Convert cassava tonnage to MOCAF flour-equivalent at
the yield rates from the deck's processing research, then express it as a share of Indonesia's wheat
consumption. **These are ceilings** (all cassava → flour, which won't happen — cassava has food/feed/starch
uses today), so they bound the opportunity rather than predict it.

MOCAF conversion yields (from the deck / processing studies): community-scale sun-dried ~20%, lab tape-yeast ~34.6%.

Wheat consumption reference: USDA GAIN ID2026-0010 — food-wheat ~9.8 MMT, total ~11.6 MMT (2025/26).

In [ ]:
cassava_prod_t   = cpiv['Production'].loc[LATEST]          # tonnes, latest
MOCAF_YIELDS     = {'community sun-dried (~20%)': 0.20, 'lab tape-yeast (~34.6%)': 0.346}
WHEAT_FOOD_MMT   = 9.8    # USDA GAIN, food use 2025/26
WHEAT_TOTAL_MMT  = 11.6   # USDA GAIN, total consumption 2025/26

print(f'Cassava production, {LATEST}: {cassava_prod_t/1e6:,.2f} Mt (fresh roots)\n')
print(f"{'MOCAF conversion':28s} {'flour-equiv (Mt)':>16s} {'% food wheat':>14s} {'% total wheat':>14s}")
for label, y in MOCAF_YIELDS.items():
    flour_mmt = cassava_prod_t * y / 1e6
    print(f'{label:28s} {flour_mmt:>16,.2f} {100*flour_mmt/WHEAT_FOOD_MMT:>13,.0f}% {100*flour_mmt/WHEAT_TOTAL_MMT:>13,.0f}%')

print('\nReality check: cassava is already consumed (food, animal feed, starch/tapioca), so the'
      ' realistic substitution share is a FRACTION of these ceilings. Pair with a 10–20% blend'
      ' rate to get the achievable band for slide 8.')

In [ ]:
# Achievable band: apply a realistic blend rate to wheat FOOD demand, and back out the
# cassava-flour (hence fresh-cassava) tonnage that blend would require. This is the
# 'what would it actually take' framing for the recommendation.
for blend in [0.10, 0.20]:
    flour_needed_mmt = WHEAT_FOOD_MMT * blend
    for label, y in MOCAF_YIELDS.items():
        fresh_needed_mmt = flour_needed_mmt / y
        share_of_curr = 100 * fresh_needed_mmt / (cassava_prod_t/1e6)
        print(f'{int(blend*100)}% blend  |  {label:28s} -> needs {flour_needed_mmt:,.2f} Mt flour '
              f'= {fresh_needed_mmt:,.2f} Mt fresh cassava ({share_of_curr:,.0f}% of {LATEST} crop)')
    print()

## 7. Findings — feed into notebook 04 synthesis

Fill the bracketed values from the outputs above.

**The GROW production-side case for the wheat thesis:**
- **Structural zero:** Indonesia produces *no* wheat (0 QCL records, all years) while growing rice, maize,
  and cassava at scale — the production-side root of total import dependence.
- **Cassava base:** production changed **[+/− X%]** 2010→latest, driven by **[area / yield]** — so the MOCAF
  raw-material base is **[shrinking / holding / growing]**. *(This is the tension: the proposed solution's
  input may itself be under pressure.)*
- **Yield headroom:** Indonesia sits **[X%]** below the cassava frontier ([country]); closing it would add
  **[X Mt]** on today's area — substitution capacity from intensification, no land conversion.
- **Offset ceiling:** at latest production, cassava → MOCAF could in principle cover **[X–Y%]** of food-wheat
  demand; a realistic 10–20% blend would need **[X Mt]** of fresh cassava (**[X%]** of the current crop).

**Recommendation (replaces nb 04's generic go/no-go):**
> Cassava is the credible domestic lever against wheat-import vulnerability, but its constraint is
> **[area decline / yield gap]**, not raw availability. Policy leverage is therefore **[yield / processing /
> land-retention]**—focused, matching the MOCAF processing + farmer-regeneration barriers the SUSTAIN
> research already identified. The GROW data both *motivates* substitution (structural wheat zero) and
> *bounds* it (cassava base + yield gap).

In [ ]:
print('Notebook 05 complete. Carry the bracketed numbers into 04_synthesis and slide 8.')